# Демо: парсер Wildberries

Этот ноутбук позволяет запустить и протестировать парсер WB интерактивно.

**Запуск:** `uv run jupyter lab` из корня проекта, затем открыть этот файл.

Запускай ячейки по порядку через **Shift+Enter**.

In [1]:
import os
import sys
from pathlib import Path

# Если Jupyter запустился из папки notebooks/ — переходим в корень проекта.
# Это нужно, чтобы папки output/ и logs/ создавались в корне, а не здесь.
if Path.cwd().name == 'notebooks':
    os.chdir('..')

# Добавляем корень проекта в путь поиска модулей
project_root = str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f'Рабочая директория: {Path.cwd()}')

Рабочая директория: D:\Projects\interica


In [2]:
from src.logger import setup_logging

# Настраиваем логирование. Можно запускать повторно — дублей не будет.
setup_logging()
print('Логирование настроено → консоль + logs/app.log')

2026-07-12 12:39:14 [INFO    ] src.logger:setup_logging - Система логирования инициализирована: D:\Projects\interica\logs


Логирование настроено → консоль + logs/app.log


In [3]:
from src.config import settings

print(f'DEMO_MODE:    {settings.demo_mode}')
print(f'Лимит:        {settings.demo_limit if settings.demo_mode else settings.wb_max_products} товаров')
print(f'Запрос к WB:  {settings.wb_query}')
print(f'Выходная папка: {settings.output_dir}')

DEMO_MODE:    False
Лимит:        10000 товаров
Запрос к WB:  смартфон
Выходная папка: output


In [4]:
from src.parser.wb import fetch_products

# Jupyter поддерживает await напрямую — asyncio.run() не нужен.
# fetch_products() сам читает DEMO_MODE из конфига.
products = await fetch_products()
print(f'\nИтого собрано: {len(products)} товаров')

2026-07-12 12:39:25 [INFO    ] src.parser.wb:fetch_products - Страница 1: +100 товаров (итого 100 / 10000)
2026-07-12 12:39:28 [INFO    ] src.parser.wb:fetch_products - Страница 2: +100 товаров (итого 200 / 10000)
2026-07-12 12:39:30 [INFO    ] src.parser.wb:fetch_products - Страница 3: +100 товаров (итого 300 / 10000)
2026-07-12 12:39:33 [INFO    ] src.parser.wb:fetch_products - Страница 4: +100 товаров (итого 400 / 10000)
2026-07-12 12:39:35 [WARNING ] src.parser.wb:_log_retry - Rate limit 429 — попытка 1, ждём перед повтором...
2026-07-12 12:39:36 [WARNING ] src.parser.wb:_log_retry - Rate limit 429 — попытка 2, ждём перед повтором...
2026-07-12 12:39:38 [INFO    ] src.parser.wb:fetch_products - Страница 5: +100 товаров (итого 500 / 10000)
2026-07-12 12:39:41 [WARNING ] src.parser.wb:_log_retry - Rate limit 429 — попытка 1, ждём перед повтором...
2026-07-12 12:39:42 [WARNING ] src.parser.wb:_log_retry - Rate limit 429 — попытка 2, ждём перед повтором...
2026-07-12 12:39:45 [INFO    


Итого собрано: 1900 товаров


In [8]:
import json

# Смотрим структуру первого товара целиком (сырой JSON от WB)
print(json.dumps(products[0], ensure_ascii=False, indent=2))

{
  "id": 523322228,
  "__sort": 34738,
  "ksort": 0,
  "root": 529789241,
  "kindId": 0,
  "brand": "Samsung",
  "brandId": 5772,
  "siteBrandId": 0,
  "colors": [
    {
      "name": "черный",
      "id": 0
    }
  ],
  "subjectId": 515,
  "subjectParentId": 6258,
  "semanticId": [
    106,
    1,
    70,
    116,
    238,
    600,
    918
  ],
  "name": "Смартфон Galaxy A07 4 128Gb LTE DS black (ru)",
  "entity": "",
  "matchId": 346734734,
  "supplier": "MTS Shop",
  "supplierId": 509363,
  "supplierRating": 4.9,
  "supplierFlags": 12224,
  "pics": 8,
  "rating": 5,
  "reviewRating": 4.9,
  "nmReviewRating": 4.9,
  "feedbacks": 144,
  "nmFeedbacks": 144,
  "panelPromoId": 1047161,
  "volume": 5,
  "weight": 0.26,
  "viewFlags": 135553033,
  "mtype": 5,
  "sizes": [
    {
      "name": "",
      "origName": "0",
      "rank": 0,
      "optionId": 722459832,
      "wh": 507,
      "time1": 2,
      "time2": 22,
      "dtype": 6597069766664,
      "price": {
        "basic": 920000,
 

In [5]:
# Краткая сводка: id / название / бренд / цена / рейтинг
print(f'{'ID':<12} {'Название':<45} {'Бренд':<15} {'Цена, руб':>10} {'Рейтинг':>8}')
print('-' * 100)
for p in products[:10]:
    sizes = p.get('sizes', [])
    price_raw = sizes[0].get('price', {}).get('product', 0) if sizes else 0
    price = price_raw / 100
    name = p.get('name', '')[:44]
    print(f"{p['id']:<12} {name:<45} {p.get('brand', ''):<15} {price:>10.0f} {p.get('reviewRating', ''):>8}")

ID           Название                                      Бренд            Цена, руб  Рейтинг
----------------------------------------------------------------------------------------------------
523322228    Смартфон Galaxy A07 4 128Gb LTE DS black (ru  Samsung               6670      4.9
649967392    Смартфон P3 Lite 4 128 ГБ, зеленая сосна      Realme                7839      4.9
642258559    Смартфон SPARK 40 Pro 256+8GB Ink Black       TECNO                14553      4.9
1014923366   C81 Pro чёрный 4G RAM 256G ROM                POCO                  9348        5
1124200171   Смартфон Spark Go 3 4 64 ГБ (Черный)          TECNO                 7018        5
649967393    Смартфон P3 Lite 4 128 ГБ, белое облако       Realme                7839      4.9
313224783    Note 60X 3+64GB                               Realme                7514      4.8
581461952    P3 Lite 8+256GB                               Realme               11251      4.9
1216378513   Смартфон игровой GT20 PRO 16ГБ 

In [6]:
from src.parser.wb import save_raw

path = save_raw(products)
print(f'Файл: {path}')

2026-07-12 12:24:46 [INFO    ] src.parser.wb:save_raw - Сохранено 500 товаров → output\wb_raw_20260712T092446.json


Файл: output\wb_raw_20260712T092446.json
